In [0]:
from pyspark.sql import functions as F

In [0]:
df_silver = spark.table("kbeauty_silver_products")

display(df_silver)

brand,category,in_stock,key_ingredients,price_brl,product_id,product_name,rating,has_missing_price,has_missing_rating
Sulwhasoo,Skincare,true,List(Ginseng),89.9,KB004,Concentrated Ginseng Mask,null,false,true
Medicube,Skincare,true,List(PDRN),149.9,KB002,AGE-R PDRN Booster Gel,4.7,false,false
Iope,Skincare,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9,true,false
Medicube,Beauty Device,true,"List(PDRN, Peptides)",1899.9,KB001,AGE-R Booster Pro X2,4.8,false,false
Cosrx,Skincare,false,List(Hydrocolloid),39.9,KB003,Acne Master Patch,4.6,false,false


In [0]:
df_gold_brand = (
    df_silver
    .groupBy("brand")
    .agg(
        F.count("*").alias("total_products"),

        F.round(
            F.avg("price_brl"),
            2
        ).alias("avg_price_brl"),

        F.round(
            F.avg("rating"),
            2
        ).alias("avg_rating"),

        F.round(
            F.avg(
                F.when(
                    F.col("in_stock") == True,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias("in_stock_percentage")
    )
    .orderBy(
        F.col("total_products").desc()
    )
)

display(df_gold_brand)

brand,total_products,avg_price_brl,avg_rating,in_stock_percentage
Medicube,2,1024.9,4.75,100.0
Sulwhasoo,1,89.9,null,100.0
Iope,1,null,4.9,100.0
Cosrx,1,39.9,4.6,0.0


In [0]:
df_gold_brand.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_brand_summary")

In [0]:
spark.sql("""
SELECT *
FROM gold_brand_summary
""").show(truncate=False)

+---------+--------------+-------------+----------+-------------------+
|brand    |total_products|avg_price_brl|avg_rating|in_stock_percentage|
+---------+--------------+-------------+----------+-------------------+
|Medicube |2             |1024.9       |4.75      |100.0              |
|Cosrx    |1             |39.9         |4.6       |0.0                |
|Iope     |1             |NULL         |4.9       |100.0              |
|Sulwhasoo|1             |89.9         |NULL      |100.0              |
+---------+--------------+-------------+----------+-------------------+



In [0]:
df_gold_category = (
    df_silver
    .groupBy("category")
    .agg(
        F.count("*").alias("total_products"),

        F.round(
            F.avg("price_brl"),
            2
        ).alias("avg_price_brl"),

        F.round(
            F.avg("rating"),
            2
        ).alias("avg_rating"),

        F.round(
            F.avg(
                F.when(
                    F.col("in_stock") == True,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias("in_stock_percentage")
    )
    .orderBy(
        F.col("total_products").desc()
    )
)

display(df_gold_category)

category,total_products,avg_price_brl,avg_rating,in_stock_percentage
Skincare,4,93.23,4.73,75.0
Beauty Device,1,1899.9,4.8,100.0


In [0]:
df_gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_category_summary")

In [0]:
spark.sql("""
SELECT *
FROM gold_category_summary
""").show(truncate=False)

+-------------+--------------+-------------+----------+-------------------+
|category     |total_products|avg_price_brl|avg_rating|in_stock_percentage|
+-------------+--------------+-------------+----------+-------------------+
|Skincare     |4             |93.23        |4.73      |75.0               |
|Beauty Device|1             |1899.9       |4.8       |100.0              |
+-------------+--------------+-------------+----------+-------------------+



In [0]:
spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").show()

+---------+-------+
|  catalog| schema|
+---------+-------+
|workspace|default|
+---------+-------+



In [0]:
spark.sql("""
SHOW TABLES
""").show(truncate=False)

+--------+-----------------------+-----------+
|database|tableName              |isTemporary|
+--------+-----------------------+-----------+
|default |gold_brand_summary     |false      |
|default |gold_category_summary  |false      |
|default |kbeauty_bronze_products|false      |
|default |kbeauty_silver_products|false      |
+--------+-----------------------+-----------+

